# Education Data Cleaning & Preparation
##Project Overview
This project focuses on cleaning and preparing educational data for visualization in Power BI.  
The dataset includes information about students, teachers, schools, principles, regions and other data in Saudi Arabia.

In [ ]:
import pandas as pd
import numpy as np

Download the required files in CSV formatt

In [ ]:
files = ["Educational_Qassim.csv", "Educational_Riyadh.csv", "Educational_Tabuk.csv", "Educational_Hail.csv", "Educational_AlAhsa.csv", "Educational_AlJouf.csv", "Educational_Eastern.csv", "Educational_Northen_Borders.csv", "Educational_Makkah.csv", "Educational_Madinah.csv"]

## Data Cleaning
- Convert the Arabic data to English
- Converting Arabic numbers to English


In [ ]:



def clean_data(file):
    df = pd.read_csv(file)

    # 1. تنظيف أسماء الأعمدة
    df.columns = df.columns.str.strip()

    df.columns = [
        "year_gregorian",
        "year_hijri",
        "region",
        "stage",
        "authority",
        "school_gender",
        "principals",
        "teachers",
        "administrators",
        "classes",
        "students",
        "schools"
    ]

    # 2. ترجمة القيم

    df["authority"] = df["authority"].replace({
        "حكومي": "Public",
        "أهلي": "Private",
      	"عالمي وأجنبي": "International",
        "الهيئة الملكية": "Royal Commission Schools"
    })

    df["stage"] = df["stage"].replace({
      	"المرحلة الإبتدائية": "Primary",
        "المرحلة المتوسطة": "Middle",
        "المرحلة الثانوية": "Secondary",
        "رياض الأطفال": "Kindergarten"
    })

    df["school_gender"] = df["school_gender"].replace({
        "بنين": "Boys",
        "بنات": "Girls"
    })

    # 3. تحويل الأرقام العربية

    def convert_arabic(col):
        # Convert Arabic numerals to standard digits
        temp_col_str = col.astype(str).replace(
            ['٠','١','٢','٣','٤','٥','٦','٧','٨','٩'],
            ['0','1','2','3','4','5','6','7','8','9'],
            regex=True
        )
        # Convert to numeric, coercing errors to NaN
        numeric_col = pd.to_numeric(temp_col_str, errors='coerce')

        # Replace inf/-inf with NaN, then fill all NaNs with 0, then convert to int
        return numeric_col.replace([np.inf, -np.inf], np.nan).fillna(0).astype(int)

    numeric_cols = [
        "principals", "teachers", "administrators",
        "classes", "students", "schools"
    ]

    for col in numeric_cols:
        df[col] = convert_arabic(df[col])

    return df

## Append all the files in one dataset

In [ ]:

df_list = []

# df = pd.read_csv("Educational_Qassim.csv")
# print(df.columns)

for file in files:
    cleaned_df = clean_data(file)
    df_list.append(cleaned_df)

df = pd.concat(df_list, ignore_index=True)

In [ ]:
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 217 entries, 0 to 216
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   year_gregorian  214 non-null    float64
 1   year_hijri      214 non-null    float64
 2   region          214 non-null    object 
 3   stage           214 non-null    object 
 4   authority       214 non-null    object 
 5   school_gender   214 non-null    object 
 6   principals      217 non-null    int64  
 7   teachers        217 non-null    int64  
 8   administrators  217 non-null    int64  
 9   classes         217 non-null    int64  
 10  students        217 non-null    int64  
 11  schools         217 non-null    int64  
dtypes: float64(2), int64(6), object(4)
memory usage: 20.5+ KB


In [ ]:
df["year_gregorian"] = df["year_gregorian"].astype("Int64")
df["year_hijri"] = df["year_hijri"].astype("Int64")

In [ ]:
df["student_teacher_ratio"] = df["students"] / df["teachers"]
df["students_per_class"] = df["students"] / df["classes"]
df["students_per_school"] = df["students"] / df["schools"]

df.to_csv("Final_ready_data.csv", index=False)

In [ ]:
df = pd.read_csv("Final_ready_data.csv")

- Reconvert the Arabic number to make sure every thing converted correcrtly
- Renaming regions to English


In [ ]:
arabic_to_english = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")

def convert_arabic_numbers(col):
    return col.astype(str).str.translate(arabic_to_english)

In [ ]:
df["year_gregorian"] = convert_arabic_numbers(df["year_gregorian"])
df["year_hijri"] = convert_arabic_numbers(df["year_hijri"])

In [ ]:
df["year_gregorian"] = pd.to_numeric(df["year_gregorian"], errors="coerce").astype("Int64")
df["year_hijri"] = pd.to_numeric(df["year_hijri"], errors="coerce").astype("Int64")

In [ ]:
region_map = {
    "الرياض": "Riyadh",
    "القصيم": "Qassim",
    "تبوك": "Tabuk",
    "حائل": "Hail",
    "الاحساء": "Alahsa",
    "الشرقية": "Eastern",
    "الحدود الشمالية": "Northern Borders",
    "الجوف": "Aljouf",
    "المدينة المنورة": "Madinah",
    "مكة المكرمة": "Makkah"
}

In [ ]:
df["region"] = df["region"].replace(region_map)

In [ ]:
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = convert_arabic_numbers(df[col])

In [ ]:
df.head()

,year_gregorian,year_hijri,region,stage,authority,school_gender,principals,teachers,administrators,classes,students,schools,student_teacher_ratio,students_per_class,students_per_school
0,2025,1446,Qassim,Kindergarten,Public,Girls,51,873,649,711,14604,274,16.728522,20.540084,53.299270
1,2025,1446,Qassim,Kindergarten,Private,Girls,27,284,71,256,3434,59,12.091549,13.414062,58.203390
2,2025,1446,Qassim,Kindergarten,International,Girls,6,127,17,91,1352,22,10.645669,14.857143,61.454545
3,2025,1446,Qassim,Primary,Public,Girls,405,8155,2458,4656,97376,469,11.940650,20.914089,207.624733
4,2025,1446,Qassim,Primary,Public,Boys,210,5860,723,2223,44168,272,7.537201,19.868646,162.382353


In [ ]:
df.dtypes

,0
year_gregorian,Int64
year_hijri,Int64
region,object
stage,object
authority,object
school_gender,object
principals,int64
teachers,int64
administrators,int64
classes,int64


In [ ]:
df.to_csv("Final_cleaned_data.csv", index=False)

In [ ]:
from google.colab import files
files.download("Final_cleaned_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>